# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/azam-hussain-ml/Starter-Notebook-flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**My unit of analysis is one content item for one client on one report date.**

I am using March 2026 as my working month, from **2026-03-01 to 2026-03-31**. I chose a mid-panel month so the final month can stay separate for later testing.

I use the `fact_content_daily_performance` warehouse table. March 2026 is my feature window, and I use the same table for April 2026 only to create the future outcome label.

I checked the grain using `report_date`, `client_hash_id`, and `content_hash_id` and found **0 duplicate rows**.

For March 2026, the data contains:

- **9,841,378 rows**
- **55 clients**
- **331,437 content items**

For my Lane 2 task, I want to rank content that may need a refresh. I will only use information that is available before the decision point, and I will check the GSC and GA4 availability flags before using those metrics.

In [5]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print("HF token loaded successfully:", HF_TOKEN is not None)

HF token loaded successfully: True


In [6]:
%pip -q install duckdb

import duckdb

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf (
    TYPE huggingface,
    TOKEN '{HF_TOKEN}'
)
""")

print("DuckDB connected to Hugging Face successfully.")

DuckDB connected to Hugging Face successfully.


In [7]:
MARCH = """
read_parquet(
  'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
"""

schema = con.sql(f"""
DESCRIBE SELECT * FROM {MARCH}
""").df()

display(schema[["column_name", "column_type"]])

,column_name,column_type
0,report_date,DATE
1,client_hash_id,VARCHAR
2,content_hash_id,VARCHAR
3,client_has_gsc,BOOLEAN
4,client_has_ga4,BOOLEAN
5,gsc_data_available,BOOLEAN
6,ga4_data_available,BOOLEAN
7,gsc_impressions,BIGINT
8,gsc_clicks,BIGINT
9,gsc_sum_position,BIGINT


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### My field contract

**Features — March 2026**

I will build my first five features from GSC information that is available during the March feature window:

- `gsc_impressions` — search visibility.
- `gsc_clicks` — search traffic received.
- `gsc_sum_position` — used to calculate position over the feature window.
- CTR will be calculated from clicks and impressions.
- Active days will be counted from days where GSC data is available.

**Label / proxy**

My proxy target is whether a content item shows a meaningful drop in search impressions in the following period. The future outcome is only used as the label, not as a feature.

**Context**

`report_date`, `client_hash_id`, and `content_hash_id` are used to define the row, time window, joins and grouping. `gsc_data_available` and `ga4_data_available` are used as availability checks. The ID fields will not be given to the model.

**Excluded**

I am leaving GA4 metrics out of this first feature frame because GA4 coverage is much smaller in this March slice. I also exclude any information from the future label window from the feature set because that would leak the answer into the model.

In [7]:
feature_source_fields = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_sum_position"
]

context_fields = [
    "report_date",
    "client_hash_id",
    "content_hash_id",
    "gsc_data_available",
    "ga4_data_available"
]

available_columns = set(schema["column_name"])

print("Feature source fields present:")
for field in feature_source_fields:
    print(field, "->", field in available_columns)

print("\nContext fields present:")
for field in context_fields:
    print(field, "->", field in available_columns)

ga4_metric_fields = [
    c for c in schema["column_name"]
    if c.startswith("ga4_") and c != "ga4_data_available"
]

print("\nGA4 metric fields excluded from first feature frame:")
print(ga4_metric_fields)

Feature source fields present:
gsc_impressions -> True
gsc_clicks -> True
gsc_sum_position -> True

Context fields present:
report_date -> True
client_hash_id -> True
content_hash_id -> True
gsc_data_available -> True
ga4_data_available -> True

GA4 metric fields excluded from first feature frame:
['ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec']


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [8]:
# Query 1 — check the grain
grain_check = con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS row_count
FROM {MARCH}
GROUP BY report_date, client_hash_id, content_hash_id
HAVING COUNT(*) > 1
LIMIT 5
""").df()

print("Query 1 — Grain check")
print("Duplicate grain rows found:", len(grain_check))
display(grain_check)


# Query 2 — row count and time window
month_summary = con.sql(f"""
SELECT
    COUNT(*) AS row_count,
    COUNT(DISTINCT client_hash_id) AS clients,
    COUNT(DISTINCT content_hash_id) AS content_items,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM {MARCH}
""").df()

print("\nQuery 2 — March 2026 count and date span")
display(month_summary)


# Query 3 — data availability
availability_check = con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available_rows,
    COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows,
    COUNT(*) FILTER (
        WHERE gsc_data_available IS TRUE
          AND ga4_data_available IS TRUE
    ) AS both_available_rows
FROM {MARCH}
""").df()

print("\nQuery 3 — Availability check")
display(availability_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Query 1 — Grain check
Duplicate grain rows found: 0


,report_date,client_hash_id,content_hash_id,row_count



Query 2 — March 2026 count and date span


,row_count,clients,content_items,first_date,last_date
0,9841378,55,331437,2026-03-01,2026-03-31



Query 3 — Availability check


,total_rows,gsc_available_rows,ga4_available_rows,both_available_rows
0,9841378,3611061,413966,364347


### Five-feature frame

For my refresh-opportunity ranking, I will use five features from March:

1. `impressions_31d` — total search impressions during March. This is known at the decision moment.
2. `clicks_31d` — total search clicks during March. This is known at the decision moment.
3. `ctr_31d` — clicks divided by impressions. It only uses March information.
4. `avg_position_31d` — average search position during March. It is measured before the future outcome.
5. `active_gsc_days` — number of March days where GSC data is available. This is known at the decision moment because March has already ended before I make the ranking.

The label is `declined_next_month`: whether April impressions fall by more than 20% compared with March. April is used only for the outcome, not as a feature.

In [11]:
APRIL = """
read_parquet(
  'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-04/*.parquet'
)
"""

feature_frame = con.sql(f"""
WITH march AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS impressions_31d,
        SUM(gsc_clicks) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS clicks_31d,
        SUM(gsc_sum_position) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS sum_position_31d,
        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS active_gsc_days
    FROM {MARCH}
    GROUP BY client_hash_id, content_hash_id
),
april AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS april_impressions,
        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS april_gsc_days
    FROM {APRIL}
    GROUP BY client_hash_id, content_hash_id
)

SELECT
    m.client_hash_id,
    m.content_hash_id,
    m.impressions_31d,
    m.clicks_31d,
    100.0 * m.clicks_31d / m.impressions_31d AS ctr_31d,
    1.0 * m.sum_position_31d / m.impressions_31d AS avg_position_31d,
    m.active_gsc_days,
    (a.april_impressions < 0.80 * m.impressions_31d) AS declined_next_month
FROM march m
JOIN april a
    USING (client_hash_id, content_hash_id)
WHERE
    m.impressions_31d >= 100
    AND m.active_gsc_days > 0
    AND a.april_gsc_days > 0
""").df()

feature_cols = [
    "impressions_31d",
    "clicks_31d",
    "ctr_31d",
    "avg_position_31d",
    "active_gsc_days"
]

print("Rows in feature frame:", len(feature_frame))
print("Number of features:", len(feature_cols))
display(feature_frame.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows in feature frame: 100893
Number of features: 5


,client_hash_id,content_hash_id,impressions_31d,clicks_31d,ctr_31d,avg_position_31d,active_gsc_days,declined_next_month
0,client_73cda7b4e4f265ea,content_7a105f548d9c6916,6523.0,7.0,0.107313,6.893301,31,False
1,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,453.0,0.0,0.000000,3.214128,31,False
2,client_73cda7b4e4f265ea,content_36c36abc7650d7af,5630.0,6.0,0.106572,6.535346,31,False
3,client_73cda7b4e4f265ea,content_a7da352b73b02668,4944.0,13.0,0.262945,7.435680,31,False
4,client_73cda7b4e4f265ea,content_1855a661b4d36130,429.0,1.0,0.233100,3.871795,31,True


### Leakage trap

To see how leakage can fool a model, I first score the model using only the five March features. Then I deliberately add a column copied from the future label. That column should not be available when the ranking decision is made.

If the score becomes unrealistically high after adding it, that is evidence of leakage. I will remove the leaked column and keep the honest score.

In [12]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

X = feature_frame[feature_cols].copy()
y = feature_frame["declined_next_month"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

# Honest model — only March features
honest_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=8,
    random_state=42,
    n_jobs=-1
)

honest_model.fit(X_train, y_train)

honest_score = roc_auc_score(
    y_test,
    honest_model.predict_proba(X_test)[:, 1]
)

print("Honest ROC-AUC:", round(honest_score, 3))


# Deliberate leakage: copy the future label into the features
X_leaky = X.copy()
X_leaky["future_label_leak"] = y

X_train_l, X_test_l, y_train_l, y_test_l = train_test_split(
    X_leaky,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

leaky_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=8,
    random_state=42,
    n_jobs=-1
)

leaky_model.fit(X_train_l, y_train_l)

leaky_score = roc_auc_score(
    y_test_l,
    leaky_model.predict_proba(X_test_l)[:, 1]
)

print("Leaky ROC-AUC:", round(leaky_score, 3))


# Remove the deliberate leak
X_leaky = X_leaky.drop(columns=["future_label_leak"])

print("Leak removed:", "future_label_leak" not in X_leaky.columns)
print("Final feature count:", len(X_leaky.columns))

Honest ROC-AUC: 0.683
Leaky ROC-AUC: 1.0
Leak removed: True
Final feature count: 5


### Leakage result

Using only the five March features, the model achieved an ROC-AUC of **0.683**.

When I deliberately added `future_label_leak`, which is copied from the future outcome, the ROC-AUC increased to **1.000**. This is not a real improvement. The model was effectively given the answer it was supposed to predict.

I removed the leaked column and kept the original five-feature set. The honest score of **0.683** is the result I would trust for this experiment.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [13]:
total = int(availability_check.loc[0, "total_rows"])
gsc = int(availability_check.loc[0, "gsc_available_rows"])
ga4 = int(availability_check.loc[0, "ga4_available_rows"])
both = int(availability_check.loc[0, "both_available_rows"])

print("March rows:", f"{total:,}")
print("GSC available:", f"{gsc:,}", f"({gsc/total:.1%})")
print("GA4 available:", f"{ga4:,}", f"({ga4/total:.1%})")
print("Both available:", f"{both:,}", f"({both/total:.1%})")

March rows: 9,841,378
GSC available: 3,611,061 (36.7%)
GA4 available: 413,966 (4.2%)
Both available: 364,347 (3.7%)


### What this data cannot tell me

The March data does not have the same coverage for every row. GSC is available for about **36.7%** of the rows, while GA4 is available for only **4.2%**. Because of this, I cannot treat missing analytics values as zero or assume that every content item has the same amount of history.

For my first feature frame, I therefore use GSC-based features and check `gsc_data_available` before using the metrics.

This analysis is useful for **directional, decision-support ranking**, but it cannot prove why a page declined or guarantee that refreshing the page will improve performance.

I also used March for the feature window and April for the outcome. This gives me one observed month-to-month relationship, so I would not assume that the same pattern will hold for every client or every season.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.